In [70]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap
from fiat_toolbox.well_being import Household
import fiat_toolbox.well_being.methods as em
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import matplotlib as mpl
import matplotlib.ticker as mticker

In [72]:
data_hurricane = pd.read_csv(f'data/census_damage_data_hurricane.csv')
data = pd.read_csv(f'data/census_damage_data.csv')
folder = "final plots"

FileNotFoundError: [Errno 2] No such file or directory: 'data/census_damage_data_hurricane.csv'

In [ ]:
data.info()

## Pre process data

#### Parameters

In [ ]:
max_recovery_time = 10
recovery_period = 10
discount_rate = 0.1

#### Whole dataset

In [ ]:
data['ave_k_str'] = data['Max Potential Damage: Structure']/ (data['#_households'])
data['ave_damage'] = data['Damage: Structure']/(data['#_households'])
data['ave_EAD'] = data['Risk (EAD)']/(data['#_households'])
data['ave_c_0'] = data['ave_i_0']
C_avg = data['ave_c_0'].mean()
c_minimum=21559 #from census data weighted average household of 3 people

In [ ]:
columns_to_drop = ['Max Potential Damage: Structure','Damage: Structure','Risk (EAD)',"GEOID",'i_0', 'owner_households','fraction',
                  'count','building_per_bg', 'ratio_damaged_buildings','geometry']
data = data.drop(columns=columns_to_drop)

In [ ]:
data['index'] = data.index
data['eps_rel'] = 0.0 #input for model zero instead of 0.01

#### Hurricane dataset

In [ ]:
data_hurricane['ave_k_str'] = data_hurricane['Max Potential Damage: Structure']/ (data_hurricane['#_households']*data_hurricane['ratio_damaged_buildings'])
data_hurricane['ave_damage'] = data_hurricane['Damage: Structure']/(data_hurricane['#_households']*data_hurricane['ratio_damaged_buildings']) 
data_hurricane['ave_EAD'] = data_hurricane['Risk (EAD)']/(data_hurricane['#_households'])
data_hurricane['ave_c_0'] = data_hurricane['ave_i_0']
data_hurricane['v_hurricane'] = data_hurricane['ave_damage'] / data_hurricane['ave_k_str'] #calculate loss ratio for hurricane IAN
C_avg_hurricane = data_hurricane['ave_c_0'].mean()
average_EAD = data_hurricane['ave_EAD'].mean()

In [ ]:
data_hurricane['eps_rel'] = 0.0 #input for model zero instead of 0.01
data_hurricane['index'] = data_hurricane.index

In [ ]:
data_hurricane = data_hurricane.drop(columns=columns_to_drop)

#### Divide data into income groups

In [ ]:
def weighted_quantiles(values, weights, quantiles):
    sorter = np.argsort(values)
    values_sorted = np.array(values)[sorter]
    weights_sorted = np.array(weights)[sorter]

    cumulative_weights = np.cumsum(weights_sorted)
    normalized_weights = cumulative_weights / cumulative_weights[-1]

    return np.interp(quantiles, normalized_weights, values_sorted)

In [ ]:
income_quantiles = weighted_quantiles(
    data['ave_i_0'],
    data['#_households'],
    [0.25, 0.5, 0.75])

In [ ]:
bins = [0, income_quantiles[0], income_quantiles[1], income_quantiles[2], float('inf')]
income_categories = ['Lowest incomes', 'Mid-low incomes', 'Mid-high incomes', 'Highest incomes']

data['income_class'] = pd.cut(data['ave_i_0'], bins=bins, labels=income_categories, right=False)
data_hurricane['income_class'] = pd.cut(data_hurricane['ave_i_0'], bins=bins, labels=income_categories, right=False)

#### Create scenarios

In [ ]:
# create all the loss ratio in the dataframe as columns
for loss in range(5, (25+1), 5):
    data[f"v_{loss}"] = loss / 100

In [ ]:
deductible = 0.15
payout_fraction = 0.85 # highest payout fraction possible = 1 - deductible

take_up_rates = {
    1: 0.55,
    2: 1.0}

for i, take_up in take_up_rates.items():
    # Premium
    data_hurricane[f'premium_{i}'] = (
        take_up * average_EAD * (1 - deductible))
    # Payout
    data_hurricane[f'payout_{i}'] = (
        data_hurricane['ave_damage'] * payout_fraction * take_up)
    # Post-insurance consumption
    data_hurricane[f'ave_c0_insurance_sc{i}'] = (
        data_hurricane['ave_i_0'] - data_hurricane[f'premium_{i}'])
    # Premium-to-income ratio
    data_hurricane[f'premium_share_income_sc{i}'] = (
        data_hurricane[f'premium_{i}'] / data_hurricane['ave_i_0']*100)

In [ ]:
data_grouped = data.groupby('income_class', observed=True, as_index=False).mean(numeric_only=True) 
data_grouped_hurricane = data_hurricane.groupby('income_class', observed=True, as_index=False).mean(numeric_only=True)

## Execute model

### create function to run scenarios

In [ ]:
def run_loss_scenario(
    data,
    C_avg,
    discount_rate,
    value_cmin,
    recovery_period,
    max_recovery_time,
    *,
    v_col=None,
    loss_pct=None,
    consumption_col='ave_c_0',
    payout_col=0.0,
    row_granularity='index'
):
    losses = {}
    objects = {}

    if v_col is None:
        if loss_pct is None:
            raise ValueError("Provide either v_col or loss_pct")
        v_col = f"v_{loss_pct}"

    for _, row in data.iterrows():
        name = row[row_granularity]

        insurance = (
            payout_col if isinstance(payout_col, (int, float))
            else row[payout_col])

        hh = Household(
            v=row[v_col],
            k_str=row['ave_k_str'],
            c0=row[consumption_col],
            insurance=insurance,
            c_avg=C_avg,
            cmin=value_cmin,
            pi=0.3,
            rho=discount_rate,
            dt=1/52,
            t_max=recovery_period)

        hh.opt_lambda(
            eps_rel=row['eps_rel'],
            rec_time_max=max_recovery_time)

        losses[name] = hh.get_losses()
        objects[name] = hh

    recovery_times = {}
    for name, obj in objects.items():
        rt = np.nan
        for method_name in ('recovery_time', 'get_recovery_time', 'time_to_recover'):
            if hasattr(obj, method_name):
                try:
                    fn = getattr(obj, method_name)
                    rt = fn() if callable(fn) else float(fn)
                    break
                except Exception:
                    pass
        recovery_times[name] = rt

    results_df = (
        pd.DataFrame(losses)
        .T
        .reset_index()
        .rename(columns={'index': row_granularity}))

    results_df['income_class'] = data['income_class'].values
    results_df['Recovery time'] = results_df[row_granularity].map(recovery_times)

    mean_df = (
        results_df
        .groupby('income_class', observed=True)
        .mean(numeric_only=True)
        .drop(columns=row_granularity, errors='ignore'))

    median_df = (
        results_df
        .groupby('income_class', observed=True)
        .median(numeric_only=True)
        .drop(columns=row_granularity, errors='ignore'))

    return results_df, mean_df, median_df, objects

#### define relative loss values

In [ ]:
loss_levels = list(range(5, 26, 5)) # list for the loop
event_type = tuple(f"{loss}%" for loss in loss_levels) # for the column event_type in results

### Run scenarios

#### Hurricane case

In [ ]:
# --- Hurricane & insurance scenarios ---
hurricane_scenarios = {
    'No insurance': dict(
        v_col='v_hurricane',
        consumption_col='ave_c_0',
        payout_col=0.0),
    'Insurance 55%': dict(
        v_col='v_hurricane',
        consumption_col='ave_c0_insurance_sc1',
        payout_col='payout_1'),
    'Insurance 100%': dict(
        v_col='v_hurricane',
        consumption_col='ave_c0_insurance_sc2',
        payout_col='payout_2')}

In [ ]:
scenario_results_hurricane = {}
error_rows_hurricane = {}

for name, cfg in hurricane_scenarios.items():

    good_results = []
    bad_rows = []

    for idx in data_hurricane.index:
        try:
            out = run_loss_scenario(
                data=data_hurricane.loc[[idx]],  # 1-row DataFrame (no transpose needed)
                v_col=cfg["v_col"],
                consumption_col=cfg["consumption_col"],
                payout_col=cfg["payout_col"],
                C_avg=C_avg_hurricane,
                discount_rate=discount_rate,
                value_cmin=0,
                recovery_period=recovery_period,
                max_recovery_time=max_recovery_time,)

            good_results.append(out[0])  # first element = results_df

        except Exception:
            bad_rows.append(data_hurricane.loc[idx])

    # combine all successful rows
    scenario_results_hurricane[name] = (
        pd.concat(good_results, ignore_index=True)
        if good_results else pd.DataFrame())

    # store failed rows
    error_rows_hurricane[name] = (
        pd.DataFrame(bad_rows)
        if bad_rows else pd.DataFrame())

In [ ]:
error_rows_hurricane

In [ ]:
data_hurricane.loc[[31, 77], '#_households'].sum()/ data_hurricane['#_households'].sum()*100

#### Relative loss scenarios

In [ ]:
scenario_results = {}
error_rows = {}

for loss in loss_levels:
    key = f"{loss}%"

    good_parts = []
    bad_parts = []

    for idx in data.index:
        try:
            out = run_loss_scenario(
                data=data.loc[[idx]],      # run on a single row
                loss_pct=loss,
                consumption_col="ave_c_0",
                payout_col=0.0,            # no insurance
                C_avg=C_avg,
                discount_rate=discount_rate,
                value_cmin=0,              # or None, depending on your function
                recovery_period=recovery_period,
                max_recovery_time=max_recovery_time)

            # If run_loss_scenario returns a DF, use it directly.
            # If it returns (results_df, ...), take the first element.
            results_df = out[0] if isinstance(out, (tuple, list)) else out

            good_parts.append(results_df)

        except Exception as e:
            # store original input row + error message (optional but useful)
            bad_row = data.loc[[idx]].copy()
            bad_row["error"] = str(e)
            bad_parts.append(bad_row)

    # combine successful results for this loss level
    scenario_results[key] = (pd.concat(good_parts, ignore_index=True) if good_parts else pd.DataFrame())

    # combine failed input rows for this loss level
    error_rows[key] = (pd.concat(bad_parts, ignore_index=True) if bad_parts else pd.DataFrame())

##### Cmin

In [ ]:
scenario_results_cmin = {}
error_rows_cmin = {}

for loss in loss_levels:
    key = f"{loss}%"

    good_parts = []
    bad_parts = []

    for idx in data.index:
        try:
            out = run_loss_scenario(
                data=data.loc[[idx]],      # run on a single row
                loss_pct=loss,
                consumption_col="ave_c_0",
                payout_col=0.0,            # no insurance
                C_avg=C_avg,
                discount_rate=discount_rate,
                value_cmin=c_minimum,              # or None, depending on your function
                recovery_period=recovery_period,
                max_recovery_time=max_recovery_time)

            # If run_loss_scenario returns a DF, use it directly.
            # If it returns (results_df, ...), take the first element.
            results_df = out[0] if isinstance(out, (tuple, list)) else out

            good_parts.append(results_df)

        except Exception as e:
            # store original input row + error message (optional but useful)
            bad_row = data.loc[[idx]].copy()
            bad_row["error"] = str(e)
            bad_parts.append(bad_row)

    # combine successful results for this loss level
    scenario_results_cmin[key] = (pd.concat(good_parts, ignore_index=True) if good_parts else pd.DataFrame())

    # combine failed input rows for this loss level
    error_rows_cmin[key] = (pd.concat(bad_parts, ignore_index=True) if bad_parts else pd.DataFrame())

## Format results

### Main losses

In [ ]:
# metrics for df
metrics = {
    "Wellbeing Loss": "Wellbeing Loss",
    "Asset Loss": "Asset Loss",
    "Recovery time": "Recovery time",
    "Consumption Loss": "Consumption Loss",
    "Utility Loss": "Utility Loss"}

In [ ]:
# index = income_category
columns = ['Asset Loss', 'Wellbeing Loss', 'perc_wellb_loss']

In [ ]:
def prep_mean_hurricane(results_df):
    df = (
        results_df
        .groupby('income_class', observed=True)
        .mean(numeric_only=True))

    total_wb = df['Wellbeing Loss'].sum()
    df['perc_wellb_loss'] = 100 * df['Wellbeing Loss'] / total_wb
    return df

In [ ]:
hurricane_means = {
    name: prep_mean_hurricane(results_df)
    for name, results_df in scenario_results_hurricane.items()}

hurricane = hurricane_means['No insurance']
hurricane_insuranceSC1 = hurricane_means['Insurance 55%']
hurricane_insuranceSC2 = hurricane_means['Insurance 100%']

In [ ]:
asset_mean = (data.groupby("income_class", observed=True)["ave_k_str"]
        .mean()
        .reset_index())

In [ ]:
def build_tidy_scenario_df(scenario_results_file, metrics, label_fn=str):
    final = []

    # mean asset value per income group from global `data`
    asset_mean = (
        data.groupby("income_class", observed=True)["ave_k_str"]
        .mean()
        .reset_index())

    for key, results_df in scenario_results_file.items():
        if results_df.empty:
            continue

        mean_df = (
            results_df
            .groupby("income_class", observed=True)
            .mean(numeric_only=True)
            .reset_index()
            .merge(asset_mean, on="income_class", how="left"))

        # add ratio
        mean_df["Asset loss / Asset Value"] = mean_df["Asset Loss"] / mean_df["ave_k_str"]*100

        # loop through all metrics + add ratio manually
        for metric_name, col in {**metrics, 
                                 "Asset loss / Asset Value": "Asset loss / Asset Value"}.items():

            df = mean_df[["income_class", col]].rename(columns={col: "value"}).copy()
            df["event_type"] = label_fn(key)
            df["metric"] = metric_name
            final.append(df)

    return pd.concat(final, ignore_index=True)

In [ ]:
tidy_df = build_tidy_scenario_df(scenario_results, metrics=metrics, label_fn=str)

In [ ]:
final_tidy_df_cmin = build_tidy_scenario_df(scenario_results_cmin, metrics=metrics, label_fn=str)

### Other losses

#### wellbeing loss / asset value

In [ ]:
asset_value_by_income = (data.groupby('income_class', observed=True)['ave_k_str'].mean())
asset_value_by_income

In [ ]:
wb_asset_ratio = final_tidy_df[final_tidy_df['metric'] == 'Wellbeing Loss'].copy()
wb_asset_ratio['value'] = wb_asset_ratio.apply(lambda r: r['value'] / asset_value_by_income.loc[r['income_category']],axis=1)*100
wb_asset_ratio['metric'] = 'Wellbeing Loss / Asset Value'

In [ ]:
wb_asset_ratio_cmin = final_tidy_df_cmin[final_tidy_df_cmin['metric'] == 'Wellbeing Loss'].copy()
wb_asset_ratio_cmin['value'] = wb_asset_ratio_cmin.apply(lambda r: r['value'] / asset_value_by_income.loc[r['income_category']],axis=1)*100
wb_asset_ratio_cmin['metric'] = 'Wellbeing Loss / Asset Value'

#### median results

In [ ]:
def build_median_all(scenario_results_file, metric, label_fn=str):
    tidy_median = []

    for key, results_df in scenario_results_file.items():
        if results_df.empty:
            continue

        median_df = (
            results_df
            .groupby("income_class", observed=True)[metric]
            .median()
            .reset_index()
            .rename(columns={metric: "value"}))

        median_df["event_type"] = label_fn(key)
        median_df["metric"] = f"Median {metric}"
        tidy_median.append(median_df)

    return pd.concat(tidy_median, ignore_index=True) if tidy_median else pd.DataFrame()

In [ ]:
median_recovery_all = build_median_all(scenario_results, metric="Recovery time", label_fn=str)

In [ ]:
median_recovery_all_cmin = build_median_all(scenario_results_cmin, metric="Recovery time", label_fn=str)

#### results all blockgroups (not grouped by income levels)

In [ ]:
def build_hh_df(scenario_results_file, metric, event_label_fn=None):
    hh_all = []

    for key, results_df in scenario_results_file.items():
        if results_df.empty:
            continue

        df = results_df[["index", "income_class", metric]].copy()

        df["event_type"] = (
            event_label_fn(key) if event_label_fn else str(key))
        df["metric"] = metric

        hh_all.append(df)

    return pd.concat(hh_all, ignore_index=True) if hh_all else pd.DataFrame()

In [ ]:
hh_recovery_df = build_hh_df(
    scenario_results,
    metric="Recovery time",
    event_label_fn=str)

In [ ]:
hh_wellbeing_df = build_hh_df(
    scenario_results,
    metric='Wellbeing Loss',
    event_label_fn=str)

In [ ]:
hh_recovery_df_cmin = build_hh_df(
    scenario_results_cmin,
    metric='Recovery time',
    event_label_fn=str)

In [ ]:
hh_hurricane_recovery = build_hh_df(
    scenario_results_hurricane,
    metric='Recovery time')

In [ ]:
hh_hurricane_recovery_ins = build_hh_df(
    scenario_results_hurricane,
    metric='Recovery time',
    event_label_fn=str)

#### find percentage of households that are unable to recover before a certain amount of years

In [ ]:
def error_rate_by_group(error_rows_dict, full_data, label="Run error (%)"):

    # detect which income column exists in your inputs
    income_col = "income_class" 

    # totals per income group (same denominator for every event_type)
    totals = full_data.groupby(income_col, observed=True).size().reset_index(name="total")

    # build one df of all errors with event_type = dict key
    err = []
    for event_type, df in error_rows_dict.items():
        if df is None or df.empty:
            continue
        tmp = df.copy()
        if income_col not in tmp.columns and "income_class" in tmp.columns:
            tmp[income_col] = tmp["income_class"]
        tmp["event_type"] = str(event_type)
        err.append(tmp[[income_col, "event_type"]])

    err_df = pd.concat(err, ignore_index=True) if err else pd.DataFrame(columns=[income_col, "event_type"])

    # count errors per income group & event_type
    err_counts = (
        err_df.groupby([income_col, "event_type"], observed=True)
              .size()
              .reset_index(name="errors"))

    # full grid of all income groups × all event types (so zeros appear)
    full_index = pd.MultiIndex.from_product(
        [totals[income_col], [str(k) for k in error_rows_dict.keys()]],
        names=[income_col, "event_type"]
    ).to_frame(index=False)

    out = (full_index
           .merge(err_counts, on=[income_col, "event_type"], how="left")
           .merge(totals, on=income_col, how="left")
           .fillna({"errors": 0}))

    out["value"] = out["errors"] / out["total"] * 100
    out = out[[income_col, "value", "event_type"]].rename(columns={income_col: "income_class"})
    out["metric"] = label
    return out

In [ ]:
error_pct_df = error_rate_by_group(error_rows_cmin, full_data=data, label="Run error (%)")

## gather all results and create a complete dataframe per scenario 

In [ ]:
final_tidy_df = pd.concat(
    [tidy_df,
        wb_asset_ratio,
         asset_ratio,
        median_recovery_all],
    ignore_index=True)

In [ ]:
final_tidy_df_cmin = pd.concat(
    [final_tidy_df_cmin,
        wb_asset_ratio_cmin,
        median_recovery_all_cmin],
    ignore_index=True)

## plot results

### define function for relative loss scenarios plots

In [ ]:
def plot_metric(final_df, metric, ylabel, min_event=None, max_event=None, ax=None, event_step=None, decimals=0,lim_y=0):
    df = final_df[final_df['metric'] == metric].copy()
    df['event_numeric'] = df['event_type'].str.rstrip('%').astype(float)

    if min_event is not None:
        df = df[df['event_numeric'] >= min_event]
    if max_event is not None:
        df = df[df['event_numeric'] <= max_event]
    if event_step is not None:
        df = df[df['event_numeric'] % event_step == 0]

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    for cat in df['income_class'].unique():
        subset = df[df['income_class'] == cat].sort_values('event_numeric')
        ax.plot(
            subset['event_numeric'],
            subset['value'],
            marker='o',
            label=cat,
            color=color_map.get(cat, 'black'))

    ax.set_xlabel('Relative asset loss')
    ax.set_ylabel(ylabel)

    ax.yaxis.set_major_formatter(
        mticker.StrMethodFormatter(f'{{x:,.{decimals}f}}'))

    ax.set_ylim(bottom=lim_y)   # ← ensures y-axis starts at 0

    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', labelsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    for spine in ax.spines.values():
        spine.set_alpha(0.5)

    x_ticks = sorted(df['event_numeric'].unique())
    x_labels = [f"{int(x)}%" for x in x_ticks]
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels)

    if ax is None:
        plt.tight_layout()
        plt.show()

In [ ]:
color_map = {
    'Lowest incomes': '#B2182B',
    'Mid-low incomes': '#fdae61',
    'Mid-high incomes': '#50B6D8',
    'Highest incomes': '#245194'}

### Plot relative loss scenarios

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
event_step_plot = 5
plot_metric(final_tidy_df,"Asset Loss","Asset loss ($)",ax=axes[0],event_step=event_step_plot,
            min_event=5,max_event=25)
plot_metric(final_tidy_df,"Wellbeing Loss / Asset Value","Wellbeing Loss / Asset Value (%)",ax=axes[1],event_step=event_step_plot, 
            min_event=5,max_event=25)
plt.tight_layout()
fig.savefig(f'{folder}/Figure_8.jpg', dpi=300, bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
plot_metric(final_tidy_df, "Median Recovery time", "Years",min_event=5, max_event=25,event_step=5,ax=axes[0])
plot_metric(final_tidy_df_cmin, "Median Recovery time", "Years",min_event=5, max_event=25,event_step=5,ax=axes[1])
plt.tight_layout()
fig.savefig(f'{folder}/Figure_9.jpg', dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_metric(error_pct_df, "Run error (%)", "Households unable to recover (%)",min_event=5,
    max_event=25, event_step=5,lim_y=-1, ax=ax)
fig.savefig(f'{folder}/Figure_10.jpg', dpi=300, bbox_inches='tight')

### Hurricane plots

#### Wellbeing & asset loss

In [ ]:
# labels for income groups for the x-axis o the plots
categories = ['Lowest \n incomes', 'Mid-low \n incomes', 'Mid-high \n incomes', 'Highest \n incomes']
colors = ['#D9D9D9', '#8C6D5A']
custom_cmap = ListedColormap(colors)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

hurricane[["Asset Loss", "Wellbeing Loss"]].plot(kind='bar',stacked=False,ax=ax,cmap=custom_cmap,width=0.6, alpha=0.6)
ax2 = ax.twinx()
hurricane[['perc_wellb_loss']].plot(kind='bar',ax=ax2,stacked=False,legend=False,alpha=0)

ax.set_ylabel('Annual household losses (US$)', fontsize=11)
ax2.set_ylabel('% of total well-being loss', fontsize=11)
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.tick_params(axis='y', labelsize=11)
ax2.tick_params(axis='y', labelsize=11)
ax.grid(True, alpha=0.3)
ax.legend(['Asset loss', 'Well-being Loss'],fontsize=11,frameon=True)
plt.xticks(range(len(categories)), categories, rotation=0, ha='center')
ax.tick_params(axis='x', labelrotation=0, labelsize=11)
for spine in ax.spines.values():
    spine.set_alpha(0.5)
for spine in ax2.spines.values():
    spine.set_alpha(0.5)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.show()
fig.savefig(f'{folder}/Figure_6.jpg', dpi=300, bbox_inches='tight')

#### Recovery times

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6)) 
binw=0.25
hh_hurricane_only = hh_hurricane_recovery[hh_hurricane_recovery['event_type'] == 'No insurance'].copy()

sns.histplot(data=hh_hurricane_only,x='Recovery time',ax=ax,binwidth=binw,alpha=0.4,color='lightgrey',edgecolor='gray',linewidth=0.5)
sns.histplot(data=hh_hurricane_only,x='Recovery time',binwidth=binw,ax=ax,kde=True, alpha=0,color='darkgray', linewidth=0)

sns.histplot(
    data=hh_hurricane_only,
    hue='income_class',
    x='Recovery time',binwidth=binw,
    ax=ax,kde=True, alpha=0.3,
    palette=color_map,# legend=True,
    linewidth=1)

for patch in ax.patches:
    patch.set_edgecolor(patch.get_facecolor())

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor='lightgrey', edgecolor='grey', label='Total', alpha=1)]

for group, color in color_map.items():
    legend_handles.append(
        Patch(facecolor=color, edgecolor=color, label=group, alpha=0.5))

ax.legend(handles=legend_handles,frameon=False,fontsize=11, handlelength=2,
    handleheight=1)
ax.grid(True, alpha=0.3)
ax.set_xlabel('Recovery time (years)',fontsize=11)   # X-axis label
ax.set_ylabel('Count',fontsize=11)
ax.tick_params(axis='y', labelsize=11)
ax.tick_params(axis='x', labelsize=11)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:.0f}'))
medians = (
    hh_hurricane_only
    .groupby("income_class")["Recovery time"]
    .median())

# Plot vertical lines
for group, median in medians.items():
    ax.axvline(
        median,
        color=color_map[group],
        linestyle="--",
        linewidth=1,
        alpha=0.7)
for spine in ax.spines.values():
    spine.set_alpha(0.5)
plt.show()
fig.savefig(f'{folder}/Figure_7.jpg', dpi=300, bbox_inches='tight')

### Insurance plots

In [ ]:
# reduction_df = (pd.DataFrame({
#         "No insurance": hurricane["Wellbeing Loss"],
#         "Insurance 55%": hurricane_insuranceSC1["Wellbeing Loss"],
#         "Insurance 100%": hurricane_insuranceSC2["Wellbeing Loss"]}))

# reduction_df["Reduction 55% (%)"] = (
#     (reduction_df["No insurance"] - reduction_df["Insurance 55%"])
#     / reduction_df["No insurance"] * 100)

# reduction_df["Reduction 100% (%)"] = (
#     (reduction_df["No insurance"] - reduction_df["Insurance 100%"])
#     / reduction_df["No insurance"] * 100)

In [ ]:
hurricane_insuranceSC1['Reduced Well-being loss'] = hurricane["Wellbeing Loss"]-hurricane_insuranceSC1['Wellbeing Loss']
hurricane_insuranceSC2['Reduced Well-being loss'] = hurricane["Wellbeing Loss"]-hurricane_insuranceSC2['Wellbeing Loss']

In [ ]:
data_grouped_hurricane['Reduced Asset loss 55%'] = data_grouped_hurricane['ave_damage']-data_grouped_hurricane['payout_1']
data_grouped_hurricane['Reduced Asset loss 100%'] = data_grouped_hurricane['ave_damage']-data_grouped_hurricane['payout_2']

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6))

n = len(hurricane_insuranceSC1)
bar_width = 0.2
indices = np.arange(n)

# payout → slightly left of center
pos_payout = indices - bar_width / 2

# 100% scenario → slightly right of center
pos_100 = indices + bar_width / 2


# Payout (left)
ax.bar(
    pos_payout,
    data_grouped_hurricane['payout_1'],
    width=bar_width,
    color=colors[0],
    alpha=0.6,
    label='Payout')
    # label='Asset loss - payout')

# 55% (center)
ax.bar(
    pos_payout,
    hurricane_insuranceSC1["Reduced Well-being loss"],
    width=bar_width,
    edgecolor=colors[1],
    facecolor='none',
    linewidth=1.5,
    linestyle='-',
    label='Reduced well-being Loss take-up rate 55%')

# 100% (right)
ax.bar(
    pos_100,
    hurricane_insuranceSC2["Reduced Well-being loss"],
    width=bar_width,
    edgecolor=colors[1],
    facecolor='none',
    linewidth=1.5,
    linestyle='--',
    label='Reduced well-being Loss take-up rate 100%')

# 100% (right)
ax.bar(
    pos_100,
    data_grouped_hurricane['payout_2'],
    width=bar_width,
    color=colors[0],
    alpha=0.6)

ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.set_ylabel('Annual household losses (US$)', fontsize=11)
ax.tick_params(axis='y', labelsize=11)
ax.tick_params(axis='x', labelsize=11)
ax.set_ylim(0, 150000)
plt.xticks(indices, categories)
ax.legend(fontsize=11, frameon=True)
for spine in ax.spines.values():
    spine.set_alpha(0.5)
plt.show()
fig.savefig(f'{folder}/Figure_11.jpg', dpi=300, bbox_inches='tight')

## Additional data exploration (not used for paper)

#### Create functions for dataframes with all blockgrouns (households), boxplots, scatter and scatter with lines (+colors)

In [ ]:
def plot_recovery_boxplot(
    df, 
    group_col='event_type', 
    value_col='Recovery time', 
    income_col='income_class',
    income_filter=None,              
    order=None,
    outlier_whis=1.5,
    jitter_strength=0.06,
    figsize=(5, 4)):

    # --- Optional income-category filtering ---
    if income_filter is not None:
        if isinstance(income_filter, (list, tuple, set)):
            df = df[df[income_col].isin(income_filter)]
        else:
            df = df[df[income_col] == income_filter]

    # Function to compute outliers
    def get_outliers(group, whis=outlier_whis):
        q1 = group.quantile(0.25)
        q3 = group.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - whis * iqr
        upper = q3 + whis * iqr
        return group[(group < lower) | (group > upper)]

    # Compute outliers
    outliers = (
        df.groupby(group_col)[value_col]
        .apply(get_outliers)
        .reset_index())

    # Prepare boxplot data
    box_data = [
        df.loc[df[group_col] == ev, value_col].dropna()
        for ev in order ]

    fig, ax = plt.subplots(figsize=figsize)

    # Boxplot (no fliers)
    ax.boxplot(
        box_data,
        widths=0.3,
        whis=outlier_whis,
        showfliers=False,
        patch_artist=False,
        boxprops=dict(linewidth=0.8),
        whiskerprops=dict(linewidth=0.8),
        capprops=dict(linewidth=0.8),
        medianprops=dict(color='darkblue',linewidth=1.2))

    # Map x positions
    x_positions = {ev: i + 1 for i, ev in enumerate(order)}

    # Add jitter to outliers
    x_jittered = (
        outliers[group_col]
        .map(x_positions)
        + np.random.uniform(-jitter_strength, jitter_strength, size=len(outliers)))

    # Scatter outliers
    ax.scatter(
        x_jittered,
        outliers[value_col],
        s=50,
        facecolors='none',
        edgecolors='black',
        alpha=0.4,
        linewidths=1)

    # Formatting
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(order)
    ax.set_xlabel("")
    ax.set_ylabel(f'{value_col} (years)')
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_hh_lines(
    df, 
    group_col='event_type', 
    value_col='Recovery time', 
    id_col='index',           # unique identifier for each household
    income_col='income_class',
    income_filter=None,              
    order=None,
    jitter_strength=0.06,
    figsize=(12, 7),
    marker_size=1,
    alpha_points=0.6,
    alpha_lines=0.2,
    edgecolor='black'):
    
    # --- Optional income-category filtering ---
    if income_filter is not None:
        if isinstance(income_filter, (list, tuple, set)):
            df = df[df[income_col].isin(income_filter)]
        else:
            df = df[df[income_col] == income_filter]

    fig, ax = plt.subplots(figsize=figsize)
    
    # Map x positions
    x_positions = {ev: i + 1 for i, ev in enumerate(order)}
    
    # Jitter x positions for points only
    x_jittered = df[group_col].map(x_positions) + np.random.uniform(-jitter_strength, jitter_strength, size=len(df))
    
    # Scatter all points
    ax.scatter(
        x_jittered,
        df[value_col],
        s=marker_size,
        facecolors='none',   # hollow circles
        edgecolors=edgecolor,
        alpha=alpha_points,
        linewidths=1 )
    
    # --- Connect lines for each household ---
    for uid, group in df.groupby(id_col):
        x_line = [x_positions[ev] for ev in order if ev in group[group_col].values]
        y_line = [group.loc[group[group_col]==ev, value_col].values[0] for ev in order if ev in group[group_col].values]
        ax.plot(x_line, y_line, color='gray', alpha=alpha_lines, linewidth=0.7)
    
    # Formatting
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(order)
    ax.set_xlabel(group_col)
    ax.set_ylabel(value_col)
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_hh_lines_colors(
    df,
    value_col='Recovery time',
    group_col='event_type',
    income_col='income_class',
    id_col='index',
    income_filter=None,
    order=None,
    figsize=(9, 5),
    dot_size=10,
    alpha_line=0.4,
    alpha_dot=0.6,
    color_map=None):
    
   # --- Default color map if none provided ---
    if color_map is None:
        categories = df[income_col].unique()
        palette = ['#2c7bb6','#7bc9e3','#fdae61','#d7191c']
        color_map = dict(zip(categories, palette))

    # --- Optional income-category filtering ---
    if income_filter is not None:
        if isinstance(income_filter, (list, tuple, set)):
            df = df[df[income_col].isin(income_filter)]
        else:
            df = df[df[income_col] == income_filter]

    # --- Determine x positions ---
    if order is None:
        order = list(df[group_col].unique())
    x_positions = {ev: i + 1 for i, ev in enumerate(order)}

    fig, ax = plt.subplots(figsize=figsize)

    # --- Plot each household ---
    for hh_id, hh_data in df.groupby(id_col):

        # Sort by event order (important!)
        hh_data = hh_data.copy()
        hh_data['x'] = hh_data[group_col].map(x_positions)
        hh_data = hh_data.sort_values('x')
        x = hh_data['x'].values
        y = hh_data[value_col].values

        # One income category per household
        income_cat = hh_data[income_col].iloc[0]
        color = color_map.get(income_cat, 'gray')

        # Line
        ax.plot(
            x, y,
            color=color,
            alpha=alpha_line,
            linewidth=0.8)

        # Dots
        ax.scatter(
            x, y,
            color=color,
            s=dot_size,
            alpha=alpha_dot)

    # --- Formatting ---
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(order)
    ax.set_xlabel(group_col)
    ax.set_ylabel(value_col)
    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

### plot bg results

In [ ]:
plot_recovery_boxplot(
    hh_recovery_df, 
    group_col='event_type', 
    value_col='Recovery time',order=event_type)

In [ ]:
plot_recovery_boxplot(
    hh_recovery_df_cmin, 
    group_col='event_type', 
    value_col='Recovery time',order=event_type)

In [ ]:
plot_hh_lines(
    hh_wellbeing_df, 
    group_col='event_type',
    # income_filter='low_incomes',
    value_col='Wellbeing Loss',order=event_type)

In [ ]:
plot_hh_lines_colors(
    hh_wellbeing_df, 
    group_col='event_type',
    # income_filter='low_incomes',
    value_col='Wellbeing Loss',order=event_type)

In [ ]:
plot_hh_lines_colors(
    hh_recovery_df_cmin, 
    group_col='event_type',
    # income_filter='Lowest incomes',
    value_col='Recovery time',order=event_type)